In [0]:
# Create or update the PRD Genie space with column_configs preserved
# Then verify by fetching the space back and checking column descriptions

import requests
import json
from types import SimpleNamespace

# --- Configuration ---
PRD_WAREHOUSE_ID = "27ce274520c5a4ac"  # Replace with your prod SQL warehouse ID
PRD_SPACE_ID = "01f17b92c67816e6b8c66cbf896f5990"  # Set to existing space_id to UPDATE, or None to CREATE
PARENT_PATH = "/Workspace/Users/hembram@dnli.com"

# --- Load exported space from JSON file ---
with open("/Workspace/Users/hembram@dnli.com/PJCA.json", "r") as f:
    _exported = json.load(f)
prd_serialized = _exported["serialized_space"]
space = SimpleNamespace(title=_exported["metadata"]["title"], description=_exported["metadata"]["description"])

# Sort tables by identifier (required by the API)
prd_serialized["data_sources"]["tables"].sort(key=lambda t: t["identifier"])

# --- Pre-flight: confirm column_configs are in the payload ---
tables_with_descriptions = 0
total_col_descriptions = 0
for t in prd_serialized["data_sources"]["tables"]:
    for col_cfg in t.get("column_configs", []):
        if col_cfg.get("description"):
            total_col_descriptions += 1
    if any(c.get("description") for c in t.get("column_configs", [])):
        tables_with_descriptions += 1

print(f"Pre-flight check:")
print(f"  Tables with column descriptions: {tables_with_descriptions}")
print(f"  Total column descriptions: {total_col_descriptions}")

# --- Build payload ---
serialized_space_str = json.dumps(prd_serialized)
payload = {
    "title": space.title,
    "description": space.description,
    "parent_path": PARENT_PATH,
    "warehouse_id": PRD_WAREHOUSE_ID,
    "serialized_space": serialized_space_str,  # Must be a JSON *string*
}

# --- Create or Update ---
def space_exists(sid):
    if not sid:
        return False
    resp = requests.get(f"https://{workspace_host}/api/2.0/genie/spaces/{sid}", headers=headers)
    return resp.status_code == 200

if space_exists(PRD_SPACE_ID):
    url = f"https://{workspace_host}/api/2.0/genie/spaces/{PRD_SPACE_ID}"
    response = requests.patch(url, headers=headers, json=payload)
    action = "Updated"
else:
    url = f"https://{workspace_host}/api/2.0/genie/spaces"
    response = requests.post(url, headers=headers, json=payload)
    action = "Created"

if response.status_code == 200:
    result = response.json()
    result_space_id = result.get("space_id", PRD_SPACE_ID or "N/A")
    print(f"\n{action} space: {result_space_id}")
else:
    print(f"\nERROR ({response.status_code}): {response.text}")
    result_space_id = None

# --- Verification: Fetch space back and confirm column_configs persisted ---
if result_space_id:
    print(f"\n--- Verification: fetching space back ---")
    verify_resp = requests.get(
        f"https://{workspace_host}/api/2.0/genie/spaces/{result_space_id}",
        headers=headers,
        params={"include_serialized_space": "true"},
    )
    if verify_resp.status_code == 200:
        verify_data = verify_resp.json()
        verify_serialized = json.loads(verify_data.get("serialized_space", "{}"))
        verify_tables = verify_serialized.get("data_sources", {}).get("tables", [])

        found_descriptions = 0
        for t in verify_tables:
            for col_cfg in t.get("column_configs", []):
                if col_cfg.get("description"):
                    found_descriptions += 1

        print(f"  Column descriptions in created space: {found_descriptions}")
        if found_descriptions == total_col_descriptions:
            print(f"  ✓ All {total_col_descriptions} column descriptions preserved!")
        elif found_descriptions == 0:
            print(f"  ✗ NO column descriptions found — API dropped them.")
            print(f"    Possible causes:")
            print(f"    1. Warehouse cannot access the prod tables (check permissions)")
            print(f"    2. Column names in config don't match actual table columns")
            print(f"    3. Try UPDATE after CREATE (2-step approach)")
        else:
            print(f"  ⚠ Partial: expected {total_col_descriptions}, got {found_descriptions}")
            # Show which tables lost descriptions
            for t in verify_tables:
                tid = t["identifier"]
                desc_count = sum(1 for c in t.get("column_configs", []) if c.get("description"))
                orig = next((ot for ot in prd_serialized["data_sources"]["tables"] if ot["identifier"] == tid), None)
                orig_count = sum(1 for c in orig.get("column_configs", []) if c.get("description")) if orig else 0
                if desc_count != orig_count:
                    print(f"    {tid}: expected {orig_count}, got {desc_count}")
    else:
        print(f"  Verification GET failed: {verify_resp.status_code}")